# Preprocessing

In [3]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

from config import (
    OPENROUTER_API_KEY, OPENROUTER_BASE_URL, LLM_MODEL,
    EMBEDDING_MODEL, EMBEDDING_DIMENSIONS, CHUNK_SIZE, CHUNK_OVERLAP,
)
from llm_setup import llm_model, llm_response

In [4]:
import urllib.request

from langchain_text_splitters import CharacterTextSplitter

from langchain_core.documents import Document

from langchain_openai import OpenAIEmbeddings

from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.messages import HumanMessage, AIMessage


## Load The Document

In [5]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Download the file using urllib
if os.path.exists(filename):
    os.remove(filename)
urllib.request.urlretrieve(url, filename)
print('file downloaded')

with open(filename, 'r') as file:
    contents = file.read()
    print(contents[:1000])

file downloaded
1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potentia

## Splitting the document into chunks

In [6]:
with open(filename) as f:
    text = f.read()
documents = [Document(page_content=text)]
text_splitter = CharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = text_splitter.split_documents(documents)
print(len(chunks))


Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16


## Embedding and storing


In [7]:
texts = [chunk.page_content for chunk in chunks]

openai_embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, openai_api_key=OPENROUTER_API_KEY, openai_api_base=OPENROUTER_BASE_URL, dimensions=EMBEDDING_DIMENSIONS)
chroma_doc_search = Chroma.from_texts(texts, openai_embeddings, collection_name="company-policies")
print(chroma_doc_search.get())

{'ids': ['c422e4d7-ef01-4aff-812f-cfc278cef626', '9b00115e-a422-440b-8e83-28d56f0b70c4', '10e574bc-6e1e-45ee-af28-2ed3a2883c5a', 'b3d261e0-c892-4b90-9caa-1f092117f7a1', 'f0c47bf0-32b7-4dda-a0b8-e0b75a919401', '1003b562-5ae8-41c2-a509-4dab6b3a9d5a', 'dea6caa9-0ab4-4e2a-8593-873b12d207a4', '4028f5d9-08fe-4f01-be38-7d8888f5d8a1', '55c859da-0ed8-4e75-98a3-f76255834975', '52e11786-ab4b-4b98-a872-bfe5830bf093', '580d9702-c3fa-4328-af78-52b522efb53e', '6fbff791-f80d-4e20-b93a-0abaec3a6799', '12e48707-0039-4a22-a93c-5c73689774ae', '7cc387b2-3a39-49a4-8c13-786552787831', 'cc60740b-b78e-4e5f-8fc4-d857903fba5e', '6bd2c37b-5ab4-4e10-9256-d6199fa85417'], 'embeddings': None, 'documents': ['1.\tCode of Conduct', "Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This

# LLM Model Construction

## Model

In [ ]:
#params = {
#    "temperature": 0.8,
#    "max_tokens": 1024,
#    "max_completion_tokens": 512,
#}

model = llm_model(params=None)

## Integrating Langchain

In [10]:
# already using prompt template, exercise was pushing for RetrievalQA, wich is deprecated

retriever = chroma_doc_search.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Use the given context to answer the question. If you don't know, say you don't know. Context: {context}"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(model, prompt)
qa = create_retrieval_chain(retriever, question_answer_chain)

query = "what is mobile policy?"
result = qa.invoke({"input": query})

print(result["answer"])

The Mobile Phone Policy outlines the standards and expectations for the appropriate and responsible use of mobile devices within the organization. Its purpose is to ensure that employees use mobile phones in a way that aligns with company values and complies with legal requirements. Key components of the policy include:

- **Acceptable Use**: Mobile devices should primarily be used for work-related tasks, with limited personal use allowed if it doesn't interfere with work.
- **Security**: Employees must protect their mobile devices and credentials, be cautious when downloading apps or clicking links, and report any security concerns.
- **Confidentiality**: Sensitive company information should not be transmitted via unsecured apps or emails, and discussions about company matters should be discreet in public.
- **Cost Management**: Employees should keep personal usage separate from company accounts and reimburse the company for any personal charges on company-issued phones.
- **Complianc

# Dive Deeper

## Prompt Template

Already created PromptTemplate previsously, exercise was pushing for RetrievalQA, wich is deprecated

In [11]:
query = "Can I eat in company vehicles?"
result = qa.invoke({"input": query})

print(result["answer"])

The provided context does not explicitly mention whether eating in company vehicles is allowed or not. Therefore, I don't know if eating in company vehicles is permitted according to the policies outlined.


## Make the conversation have memory

As the model does not have memory, it won't relate to it as the car
Sample response:

```Based on the Internet and Email Policy, you cannot:
1. Use company-provided internet and email services for non-job-related tasks during work hours, except for limited personal use during non-work hours that does not interfere with work responsibilities.
2. Share your login credentials or passwords with others.
3. Open email attachments or click on links from unknown sources without exercising caution.
4. Transmit confidential information, trade secrets, or sensitive customer data via email without using encryption.
5. Discuss company matters on public forums or social media without discretion.
6. Engage in harassment, discrimination, or distribute offensive or inappropriate content.
7. Violate relevant laws and regulations regarding internet and email usage, including copyright and data protection.
8. Use company internet and email in ways that may lead to disciplinary measures or termination for policy violations.
It is important to adhere to these guidelines to ensure responsible and secure usage of digital communication tools.
´´´

In [12]:
query = "What I cannot do in it?"
result = qa.invoke({"input": query})

print(result["answer"])

Based on the context provided, here are some actions you cannot do under the Internet and Email Policy and the Mobile Phone Policy:

**Internet and Email Policy:**
1. Use company-provided internet and email services for non-job-related tasks during work hours.
2. Share your login credentials or passwords with others.
3. Open email attachments or click on links from unknown sources without caution.
4. Transmit confidential information or sensitive customer data via unencrypted emails.
5. Engage in harassment, discrimination, or distribute offensive or inappropriate content through email or internet usage.
6. Violate copyright or data protection laws related to internet and email usage.
7. Ignore unusual online activity or potential security breaches without reporting them.

**Mobile Phone Policy:**
1. Use mobile devices for non-work-related tasks that disrupt work obligations during work hours.
2. Share access credentials for your mobile device.
3. Download applications or click on link

In [13]:
# Rewrite follow-up questions into standalone search queries
# so the retriever can handle references like "it"
contextualize_q_system_prompt = (
    "Given the chat history and the latest user question, "
    "formulate a standalone question that can be understood "
    "without the chat history. Do NOT answer the question; "
    "just rewrite it if needed, otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(
    model,
    chroma_doc_search.as_retriever(),
    contextualize_q_prompt,
)

# Answer using only retrieved context
qa_system_prompt = (
    "Use the given retrieved context to answer the user's question. "
    "If you don't know, say you don't know.\n\n"
    "Context: {context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(model, qa_prompt)
qa_with_memory = create_retrieval_chain(history_aware_retriever, question_answer_chain)

chat_history = []


In [14]:
query = "what is the mobile policy?"
result = qa_with_memory.invoke({
    "input": query,
    "chat_history": chat_history,
})
print(result["answer"])
chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=result["answer"]),
])

query = "Can i eat in company vehicles?"
result = qa_with_memory.invoke({
    "input": query,
    "chat_history": chat_history,
})
print(result["answer"])
chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=result["answer"]),
])

query = "What I cannot do in it?"
result = qa_with_memory.invoke({
    "input": query,
    "chat_history": chat_history,
})
print(result["answer"])
chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=result["answer"]),
])


The Mobile Phone Policy outlines the standards and expectations for the appropriate and responsible use of mobile devices within the organization. Key components include:

- **Acceptable Use**: Mobile devices should primarily be used for work-related tasks, with limited personal usage allowed as long as it does not disrupt work obligations.
- **Security**: Employees must safeguard their devices and credentials, be cautious with app downloads and links, and report any security concerns.
- **Confidentiality**: Sensitive company information should not be transmitted via unsecured apps or emails, and discussions about company matters should be discreet.
- **Cost Management**: Personal phone usage should be kept separate from company accounts, and employees must reimburse the company for any personal charges on company-issued phones.
- **Compliance**: Employees must adhere to laws and regulations regarding mobile phone usage, including data protection and privacy laws.
- **Lost or Stolen De

## Putting It All Together

Above, we built a conversational RAG pipeline step by step:
- **Chunking & Embedding**: Split the document into chunks and stored vector embeddings in Chroma.
- **Basic RAG Chain**: Retrieved relevant chunks and answered questions using LCEL.
- **Conversation Memory**: Added a history-aware retriever so follow-up questions work correctly.

Now we wrap everything into an interactive agent loop that maintains its own conversation history.

## Wrap it and make it an agent

In [15]:
def qa_agent():
    qa_agent_chat_history = []
    while True:
        qa_agent_query = input("Question: ")
        if qa_agent_query.lower() in ["quit", "exit", "bye"]:
            print("Answer: Goodbye!")
            break
        qa_agent_result = qa_with_memory.invoke({
            "input": qa_agent_query,
            "chat_history": qa_agent_chat_history,
        })
        print("Answer:", qa_agent_result["answer"])
        qa_agent_chat_history.extend([
            HumanMessage(content=qa_agent_query),
            AIMessage(content=qa_agent_result["answer"]),
        ])


In [16]:
qa_agent()  

Answer: The Mobile Phone Policy outlines the standards and expectations for the appropriate and responsible use of mobile devices within the organization. Key elements of the policy include:

- **Acceptable Use**: Mobile devices should primarily be used for work-related tasks, with limited personal usage allowed as long as it does not disrupt work obligations.
- **Security**: Employees must safeguard their mobile devices and credentials, exercise caution when downloading apps or clicking links from unknown sources, and report any security concerns or suspicious activities.
- **Confidentiality**: Employees should not transmit sensitive company information through unsecured messaging apps or emails and should be discreet when discussing company matters publicly.
- **Cost Management**: Personal phone usage should be kept separate from company accounts, and employees must reimburse the company for any personal charges on company-issued phones.
- **Compliance**: Adherence to all relevant la